# Notebook 5: Visualization

**Purpose:** Generate all figures for the EviDTI reproduction study.

**Prerequisites:** Notebooks 1-4 must have been run first. This notebook reads from:
- `runs/` — TensorBoard training logs (for Figure 1)
- `runs/.../result.csv` — model output from the official evaluation engine (for Figures 2 & 3)
- `EviDTI_Predictions_Detailed.csv` — per-sample predictions with uncertainty scores (for Figures 4 & 5)

**Figures produced:**
1. **Training curves** — Train/Val Accuracy, MCC, and Loss over epochs
2. **Confusion matrix** — from `result.csv`
3. **ROC and PR curves** — from `result.csv`
4. **Uncertainty distribution + probability density** — from `EviDTI_Predictions_Detailed.csv`
5. **Uncertainty calibration** — Accuracy by confidence interval + TP/FN/FP/TN boxplot (replicating paper Fig. 3)

## Setup: Mount Drive and Install Dependencies

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
!pip install seaborn tensorboard -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

# Output directory for all saved figures
SAVE_DIR = '/content/drive/MyDrive/EviDTI_Seed1_Results/figures'
os.makedirs(SAVE_DIR, exist_ok=True)

# Path to result.csv produced by the official evaluation engine (Figures 2 & 3)
RESULT_CSV = '/content/EviDTI-main/runs/la_test1_31-03_07-26-02/result.csv'

# Path to detailed predictions CSV with uncertainty (Figures 4 & 5)
PRED_CSV = '/content/drive/MyDrive/EviDTI_Seed_1_Test_Results_Backup/EviDTI_Phase2_Predictions_Detailed.csv'

# TensorBoard run log directory (Figure 1)
RUN_DIR = '/content/EviDTI-main/runs/la_test1_31-03_02-31-10'

print(f'result.csv:       {RESULT_CSV}')
print(f'Predictions CSV:  {PRED_CSV}')
print(f'Run log dir:      {RUN_DIR}')
print(f'Figures saved to: {SAVE_DIR}')

## Figure 1: Training Curves (Accuracy, MCC, Loss)

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

def read_tb(run_dir, sub_dir):
    """Read scalar values from a TensorBoard event file in a subdirectory."""
    path = os.path.join(run_dir, sub_dir)
    ea = event_accumulator.EventAccumulator(path)
    ea.Reload()
    tag = ea.Tags()['scalars'][0]
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

steps_acc_t,  acc_t  = read_tb(RUN_DIR, 'Acc_train')
steps_acc_v,  acc_v  = read_tb(RUN_DIR, 'Acc_val')
steps_mcc_t,  mcc_t  = read_tb(RUN_DIR, 'MCC_train')
steps_mcc_v,  mcc_v  = read_tb(RUN_DIR, 'MCC_val')
steps_loss_t, loss_t = read_tb(RUN_DIR, 'Loss_Train')
steps_loss_v, loss_v = read_tb(RUN_DIR, 'Loss_Val')

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 6))

ax1.plot(steps_acc_t, acc_t, label='Train', color='#3498db', alpha=0.8)
ax1.plot(steps_acc_v, acc_v, label='Val',   color='#e74c3c', linewidth=2)
ax1.set_title('Accuracy Comparison', fontsize=14)
ax1.set_xlabel('Steps'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(steps_mcc_t, mcc_t, label='Train', color='#f1c40f', alpha=0.8)
ax2.plot(steps_mcc_v, mcc_v, label='Val',   color='#9b59b6', linewidth=2)
ax2.set_title('MCC Comparison', fontsize=14)
ax2.set_xlabel('Steps'); ax2.set_ylabel('MCC')
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.5)

ax3.plot(steps_loss_t, loss_t, label='Train', color='#2ecc71', alpha=0.8)
ax3.plot(steps_loss_v, loss_v, label='Val',   color='#e67e22', linewidth=2)
ax3.set_title('Loss Comparison', fontsize=14)
ax3.set_xlabel('Steps'); ax3.set_ylabel('Loss')
ax3.legend(); ax3.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig1_training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## Figure 2: Confusion Matrix

Data source: `result.csv` from the official EviDTI evaluation engine.

In [ ]:
df = pd.read_csv(RESULT_CSV)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df_t = df.T

# Extract predicted probabilities (column 2) and true labels (column 3)
probs = pd.to_numeric(df_t.iloc[:, 2], errors='coerce').values

def parse_onehot_label(val):
    """Parse one-hot encoded labels stored as strings e.g. '[0 1]' or '[1 0]'."""
    val_str = str(val).replace(' ', '').replace('\n', '')
    if val_str in ['[01]', '[1]']: return 1
    if val_str in ['[10]', '[0]']: return 0
    if '1' in val_str: return 1
    return 0

true_labels = np.array([parse_onehot_label(x) for x in df_t.iloc[:, 3]])
preds       = (probs >= 0.5).astype(int)

cm = confusion_matrix(true_labels, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['True Negative', 'True Positive'],
            annot_kws={'size': 16})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix (DrugBank Test Set)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig2_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## Figure 3: ROC Curve and Precision-Recall Curve

Data source: `result.csv` from the official EviDTI evaluation engine.

In [ ]:
# probs and true_labels already loaded from Figure 2 cell above
fpr, tpr, _          = roc_curve(true_labels, probs)
roc_auc              = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(true_labels, probs)
pr_auc               = auc(recall, precision)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(fpr, tpr, color='#D32D41', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='#1F3F49', lw=2, linestyle='--', label='Random classifier')
ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('Receiver Operating Characteristic', fontsize=14)
ax1.legend(loc='lower right', fontsize=11)
ax1.grid(alpha=0.3)

ax2.plot(recall, precision, color='#6AB187', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14)
ax2.legend(loc='lower left', fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig3_roc_pr_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## Figure 4: Uncertainty Distribution and Probability Density

Data source: `EviDTI_Predictions_Detailed.csv` (contains per-sample uncertainty scores).

In [ ]:
df2 = pd.read_csv(PRED_CSV)
df2['pred']    = (df2['Predicted_Probability'] >= 0.5).astype(int)
df2['correct'] = (df2['pred'] == df2['True_Label']).map({True: 'Correct', False: 'Incorrect'})

sns.set_theme(style='whitegrid', font_scale=1.2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: uncertainty by correctness
sns.boxplot(ax=ax1, x='correct', y='Uncertainty', data=df2,
            palette='Set2', width=0.5)
ax1.set_title('Uncertainty Distribution: Correct vs. Incorrect Predictions', fontsize=13)
ax1.set_xlabel('Prediction Outcome', fontsize=12)
ax1.set_ylabel('Uncertainty Score', fontsize=12)

# Right: predicted probability density by true label
sns.kdeplot(ax=ax2, data=df2[df2['True_Label'] == 1],
            x='Predicted_Probability', fill=True,
            color='red', label='True Positive (Label=1)', linewidth=2)
sns.kdeplot(ax=ax2, data=df2[df2['True_Label'] == 0],
            x='Predicted_Probability', fill=True,
            color='blue', label='True Negative (Label=0)', linewidth=2)
ax2.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5,
            label='Decision boundary (0.5)')
ax2.set_title('Predicted Probability Density for Drug-Target Interactions', fontsize=13)
ax2.set_xlabel('Predicted Probability', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_xlim(0, 1)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_uncertainty_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

## Figure 5: Uncertainty Calibration

Replicates Figure 3 from the original EviDTI paper.
Data source: `EviDTI_Predictions_Detailed.csv`

- **Left (Fig. 3b):** Prediction accuracy across 20 confidence intervals sorted by uncertainty score
- **Right (Fig. 3a):** Uncertainty score distribution grouped by prediction type (TP / FN / FP / TN)

In [ ]:
# df2 already loaded from Figure 4 cell above
prob_list   = df2['Predicted_Probability'].values
true_labels2 = df2['True_Label'].values
uncertainty = df2['Uncertainty'].values
preds2      = (prob_list >= 0.5).astype(int)
correct2    = (preds2 == true_labels2).astype(int)

# Confidence interval accuracy (replicates Fig. 3b)
n_intervals   = 20
sorted_idx    = np.argsort(uncertainty)  # lowest uncertainty = highest confidence
interval_size = len(sorted_idx) // n_intervals
interval_acc, interval_labels = [], []
for i in range(n_intervals):
    start = i * interval_size
    end   = (i + 1) * interval_size if i < n_intervals - 1 else len(sorted_idx)
    idx   = sorted_idx[start:end]
    interval_acc.append(correct2[idx].mean() * 100)
    interval_labels.append(f'{i*5}-{(i+1)*5}')

# TP/FN/FP/TN groups (replicates Fig. 3a)
tp_u = uncertainty[(preds2 == 1) & (true_labels2 == 1)]
fn_u = uncertainty[(preds2 == 0) & (true_labels2 == 1)]
fp_u = uncertainty[(preds2 == 1) & (true_labels2 == 0)]
tn_u = uncertainty[(preds2 == 0) & (true_labels2 == 0)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: confidence interval accuracy curve
x = np.arange(n_intervals)
ax1.plot(x, interval_acc, 'o-', color='#E24B4A', linewidth=2, markersize=5, label='DrugBank')
ax1.fill_between(x,
                 np.array(interval_acc) - 1.5,
                 np.array(interval_acc) + 1.5,
                 alpha=0.2, color='#E24B4A')
ax1.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Random baseline')
ax1.set_xlabel('Confidence intervals (%)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy vs. Uncertainty Confidence Interval', fontsize=13)
ax1.set_xticks(x[::2])
ax1.set_xticklabels(interval_labels[::2], rotation=45, fontsize=9)
ax1.set_ylim(40, 105)
ax1.legend(fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.annotate(f'{interval_acc[0]:.1f}%\n(Top 5%)',
             xy=(0, interval_acc[0]), xytext=(2, interval_acc[0] - 10),
             fontsize=10, color='#E24B4A',
             arrowprops=dict(arrowstyle='->', color='black'))

# Right: uncertainty boxplot by prediction type
bp = ax2.boxplot([tp_u, fn_u, fp_u, tn_u],
                 labels=['TP', 'FN', 'FP', 'TN'],
                 patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, color, alpha in zip(bp['boxes'],
                                ['#2ecc71', '#e74c3c', '#e74c3c', '#2ecc71'],
                                [0.8, 0.8, 0.5, 0.5]):
    patch.set_facecolor(color)
    patch.set_alpha(alpha)
for i, data in enumerate([tp_u, fn_u, fp_u, tn_u]):
    ax2.text(i + 1, np.median(data) + 0.02, f'{np.median(data):.2f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_xlabel('Prediction outcome', fontsize=12)
ax2.set_ylabel('Uncertainty score', fontsize=12)
ax2.set_title('Uncertainty Distribution by Prediction Type', fontsize=13)
ax2.grid(True, linestyle='--', alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig5_uncertainty_calibration.png'), dpi=300, bbox_inches='tight')
plt.show()

print('Figure 5 saved.')
print(f'\nTop  5% accuracy:    {interval_acc[0]:.2f}%')
print(f'Bottom 5% accuracy:  {interval_acc[-1]:.2f}%')
print('\nMedian uncertainty by group:')
for name, vals in [('TP', tp_u), ('FN', fn_u), ('FP', fp_u), ('TN', tn_u)]:
    print(f'  {name}: {np.median(vals):.4f}  (n={len(vals)})')